# 🔬 Optuna: Controllo vs Distillazione (verdetto automatico)

Cerca il **miglior** finetuning **senza** distillazione (controllo, solo `lr`) e il **miglior** finetuning
**con** distillazione FD-CMKD (`lambda`, `high_w`, `lr`), poi li **confronta**.

- Se `best_distill < best_ctrl − margine` → la distillazione **aiuta davvero**.
- Se sono ~uguali → **feature-distill neutra** → si passa al **Piano B** (shared classifier / logit).

Dati e feature teacher caricati **una sola volta**; ogni trial è un finetuning veloce dal 40%.

## 0 — Setup (root Signformer + optuna + patch scheduler)

In [14]:
import os, sys
# cwd = code/distillation (dove sta il notebook)
DIST_ROOT = os.path.abspath(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd())!='distillation' else os.getcwd())
if os.path.basename(DIST_ROOT) != 'distillation':
    # fallback: risali finche trovi la struttura code/distillation
    p = os.getcwd()
    while p and os.path.basename(p) != 'distillation':
        p = os.path.dirname(p)
    DIST_ROOT = p
os.chdir(DIST_ROOT)
# aggiungi code/signformer al sys.path per 'import main.training'
SIGNFORMER = os.path.abspath(os.path.join(DIST_ROOT, '..', 'signformer'))
sys.path.insert(0, SIGNFORMER)      # per: import main.training as mt
sys.path.insert(0, os.path.dirname(DIST_ROOT))  # per: from distillation.distill import ...
print('cwd:', os.getcwd())
print('signformer path:', SIGNFORMER)
assert os.path.isdir(os.path.join(SIGNFORMER, 'main')), 'code/signformer/main not found'


cwd: /home/ebufi/phoenix/Signformer
optuna 4.8.0


In [15]:
# --- FIX logger: trainer robusto (make_logger None / crash su _log_parameters_list) ---
import logging, main.training as mt

def _safe_make_logger(model_dir=None, log_file="train.log", *a, **k):
    lg = logging.getLogger("sf_distill")
    lg.setLevel(logging.INFO)
    if not lg.handlers:
        lg.addHandler(logging.StreamHandler())
    lg.propagate = False
    return lg

mt.make_logger = _safe_make_logger                          # make_logger valido garantito
mt.TrainManager._log_parameters_list = lambda self: None    # salta il logging cosmetico dei parametri (è quello che crasha)
print("patch logger applicata")

patch logger applicata


## 1 — Config dello studio

In [16]:
STUDY = {
    'base_cfg':          'sign_distill.yaml',   # config base
    'student_ckpt':      './student_40.ckpt',   # Signformer 40%
    'teacher_feats_dir': './skeleton_feats',    # feature teacher
    'out_root':          './optuna',            # dove finiscono i trial

    # trial veloci
    'early_stopping_patience': 6,     # stop rapido per iterare
    'n_trials_ctrl':    6,            # trial per il CONTROLLO (cerca solo lr)
    'n_trials_distill': 15,           # trial per la DISTILLAZIONE (lambda, high_w, lr)

    # spazio di ricerca
    'lr_min': 2e-5, 'lr_max': 2e-4,
    'lambda_min': 0.1, 'lambda_max': 3.0,
    'high_w_min': 0.0, 'high_w_max': 1.0,

    'margin': 0.3,   # soglia (in punti WER) per dire "la distillazione aiuta"
    'seed': 42,
}
os.makedirs(STUDY['out_root'], exist_ok=True)
STUDY

{'base_cfg': 'distillation/sign_distill.yaml',
 'student_ckpt': './distillation/student_40.ckpt',
 'teacher_feats_dir': './distillation/skeleton_feats',
 'out_root': './distillation/optuna',
 'early_stopping_patience': 6,
 'n_trials_ctrl': 6,
 'n_trials_distill': 15,
 'lr_min': 2e-05,
 'lr_max': 0.0002,
 'lambda_min': 0.1,
 'lambda_max': 3.0,
 'high_w_min': 0.0,
 'high_w_max': 1.0,
 'margin': 0.3,
 'seed': 42}

## 2 — Carica dati + vocab + feature teacher (UNA volta)

In [17]:
import main.training as mt
import distillation.distill_trainer as DT

BASE_CFG = mt.load_config(STUDY['base_cfg'])
mt.set_seed(seed=STUDY['seed'])

# dati e vocab una sola volta (riusati da tutti i trial)
TRAIN_DATA, DEV_DATA, TEST_DATA, GLS_VOCAB, TXT_VOCAB = mt.load_data(data_cfg=BASE_CFG['data'])
print('train', len(TRAIN_DATA), '| dev', len(DEV_DATA))

# feature teacher in RAM una sola volta -> ogni trainer riusa questo dict
_TF = pickle.load(open(os.path.join(STUDY['teacher_feats_dir'], 'train.pkl'), 'rb'))
DT.load_teacher_feats = lambda path: _TF     # override: niente reload per trial
print('teacher feats:', len(_TF), '| dim:', next(iter(_TF.values())).shape[1])

SGN_DIM = (sum(BASE_CFG['data']['feature_size'])
           if isinstance(BASE_CFG['data']['feature_size'], list)
           else BASE_CFG['data']['feature_size'])

train 7094 | dev 519
teacher feats: 7096 | dim: 512


## 3 — Funzioni: costruisci config del trial + allena e ritorna il dev WER

In [18]:
def build_cfg(enabled, lr, lam=0.5, low_w=1.0, high_w=0.25, tag='t0'):
    cfg = copy.deepcopy(BASE_CFG)
    tr = cfg['training']
    tr['load_model'] = STUDY['student_ckpt']
    tr['reset_best_ckpt'] = True; tr['reset_optimizer'] = True; tr['reset_scheduler'] = True
    tr['learning_rate'] = float(lr)
    tr['early_stopping_patience'] = STUDY['early_stopping_patience']
    tr['model_dir'] = os.path.join(STUDY['out_root'], tag)
    tr['overwrite'] = True
    tr['num_valid_log'] = 1
    tr['distillation'] = {
        'enabled': bool(enabled),
        'teacher_feats_dir': STUDY['teacher_feats_dir'],
        'lambda': float(lam), 'low_w': float(low_w), 'high_w': float(high_w),
    }
    return cfg

def train_return_wer(cfg):
    """Allena (finetuning) e ritorna il MIGLIOR dev WER, senza fase di test."""
    do_rec = cfg['training'].get('recognition_loss_weight', 1.0) > 0.0
    do_trs = cfg['training'].get('translation_loss_weight', 1.0) > 0.0
    mm = cfg['data'].get('multimodal', 1.0) > 0.0
    model = mt.build_model(cfg=cfg['model'], multimodal=mm,
                           gls_vocab=GLS_VOCAB, txt_vocab=TXT_VOCAB, sgn_dim=SGN_DIM,
                           do_recognition=do_rec, do_translation=do_trs)
    trainer = DT.DistillTrainManager(model=model, config=cfg)
    trainer.train_and_validate(train_data=TRAIN_DATA, valid_data=DEV_DATA)
    wer = float(trainer.best_ckpt_score)     # dev WER (%) da minimizzare
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    return wer

print('funzioni pronte')

funzioni pronte


## 4 — Study 1: CONTROLLO (Optuna cerca solo `lr`, distillazione OFF)

In [19]:
def obj_ctrl(trial):
    lr = trial.suggest_float('lr', STUDY['lr_min'], STUDY['lr_max'], log=True)
    cfg = build_cfg(enabled=False, lr=lr, tag=f"ctrl_{trial.number}")
    return train_return_wer(cfg)

study_ctrl = optuna.create_study(direction='minimize',
                                 sampler=optuna.samplers.TPESampler(seed=STUDY['seed']))
study_ctrl.optimize(obj_ctrl, n_trials=STUDY['n_trials_ctrl'])
best_ctrl = study_ctrl.best_value
print(f"\n>>> CONTROLLO  best dev WER = {best_ctrl:.2f}  | params: {study_ctrl.best_params}")

[I 2026-07-04 22:34:45,252] A new study created in memory with name: no-name-a9ba1262-5d16-442f-8fa1-bda07fcae802


COPE:  False
[builders] SophiaG non disponibile, uso Adam come fallback


Loading model from ./distillation/student_40.ckpt
Reset optimizer.
Reset scheduler.
Reset tracking of the best checkpoint.
Distillation DISABLED — training normale.
EPOCH 1
[Epoch: 001 Step: 00007200] Batch Recognition Loss:   1.293870 => Gls Tokens per Sec:     4992 || Lr: 0.000047
Hooray! New best validation result [eval_metric]!
Saving new checkpoint.
Validation result at epoch   1, step     7200: duration: 13.4636s
	Recognition Beam Size: 1	Translation Beam Size: -1	Translation Beam Alpha: -1
	Recognition Loss: 36.46562	Translation Loss: -1.00000	PPL: -1.00000
	Eval Metric: WER
	WER 39.31	(DEL: 17.37,	INS: 2.00,	SUB: 19.94)
	BLEU-4 -1.00	(BLEU-1: -1.00,	BLEU-2: -1.00,	BLEU-3: -1.00,	BLEU-4: -1.00)
	CHRF -1.00	ROUGE -1.00
Logging Recognition and Translation Outputs
Logging Sequence: 19March_2011_Saturday_tagesschau-4690
	Gloss Reference :	MORGEN FRUEH VERSCHWINDEN LANGSAM SCHNELL NORDOST BISSCHEN WOLKE BLEIBEN TROCKEN
	Gloss Hypothesis:	MORGEN ***** VERSCHWINDEN REGION  SCHNEE  NORD

COPE:  False
[builders] SophiaG non disponibile, uso Adam come fallback


[Epoch: 001 Step: 00007200] Batch Recognition Loss:   1.323400 => Gls Tokens per Sec:     5068 || Lr: 0.000179
Hooray! New best validation result [eval_metric]!
Saving new checkpoint.
Validation result at epoch   1, step     7200: duration: 13.1457s
	Recognition Beam Size: 1	Translation Beam Size: -1	Translation Beam Alpha: -1
	Recognition Loss: 36.55619	Translation Loss: -1.00000	PPL: -1.00000
	Eval Metric: WER
	WER 40.30	(DEL: 20.23,	INS: 1.57,	SUB: 18.49)
	BLEU-4 -1.00	(BLEU-1: -1.00,	BLEU-2: -1.00,	BLEU-3: -1.00,	BLEU-4: -1.00)
	CHRF -1.00	ROUGE -1.00
Logging Recognition and Translation Outputs
Logging Sequence: 19May_2010_Wednesday_heute-1119
	Gloss Reference :	WEST FLUSS   WEST REGION NACH MITTAG BESSER SONNE MOEGLICH
	Gloss Hypothesis:	**** ZWANZIG WEST ****** NACH MITTAG BESSER SONNE MOEGLICH
	Gloss Alignment :	D    S            D                                       
[Epoch: 001 Step: 00007300] Batch Recognition Loss:   1.459445 => Gls Tokens per Sec:     4997 || Lr: 0.000179

...

COPE:  False
[builders] SophiaG non disponibile, uso Adam come fallback


[Epoch: 001 Step: 00007200] Batch Recognition Loss:   1.264724 => Gls Tokens per Sec:     5028 || Lr: 0.000029
Hooray! New best validation result [eval_metric]!
Saving new checkpoint.
Validation result at epoch   1, step     7200: duration: 13.3189s
	Recognition Beam Size: 1	Translation Beam Size: -1	Translation Beam Alpha: -1
	Recognition Loss: 36.54446	Translation Loss: -1.00000	PPL: -1.00000
	Eval Metric: WER
	WER 39.55	(DEL: 16.87,	INS: 2.24,	SUB: 20.44)
	BLEU-4 -1.00	(BLEU-1: -1.00,	BLEU-2: -1.00,	BLEU-3: -1.00,	BLEU-4: -1.00)
	CHRF -1.00	ROUGE -1.00
Logging Recognition and Translation Outputs
Logging Sequence: 18April_2011_Monday_tagesschau-1316
	Gloss Reference :	HOCH    REGION BIS  FREITAG MEISTENS WETTER FREUNDLICH SONNE TEMPERATUR STEIGEN
	Gloss Hypothesis:	DEUTSCH BIS    AUCH FREITAG STARK    VIEL   FREUNDLICH SONNE TEMPERATUR STEIGEN
	Gloss Alignment :	S       S      S            S        S                                         
[Epoch: 001 Step: 00007300] Batch Recogniti

COPE:  False
[builders] SophiaG non disponibile, uso Adam come fallback


[Epoch: 001 Step: 00007200] Batch Recognition Loss:   1.300200 => Gls Tokens per Sec:     5004 || Lr: 0.000029
Hooray! New best validation result [eval_metric]!
Saving new checkpoint.
Validation result at epoch   1, step     7200: duration: 13.4419s
	Recognition Beam Size: 1	Translation Beam Size: -1	Translation Beam Alpha: -1
	Recognition Loss: 36.59760	Translation Loss: -1.00000	PPL: -1.00000
	Eval Metric: WER
	WER 39.66	(DEL: 17.51,	INS: 2.19,	SUB: 19.96)
	BLEU-4 -1.00	(BLEU-1: -1.00,	BLEU-2: -1.00,	BLEU-3: -1.00,	BLEU-4: -1.00)
	CHRF -1.00	ROUGE -1.00
Logging Recognition and Translation Outputs
Logging Sequence: 30October_2009_Friday_tagesschau-2275
	Gloss Reference :	WIND MAESSIG WEHEN
	Gloss Hypothesis:	WIND MAESSIG WEHEN
	Gloss Alignment :	                  
[Epoch: 001 Step: 00007300] Batch Recognition Loss:   1.194950 => Gls Tokens per Sec:     4940 || Lr: 0.000029
Hooray! New best validation result [eval_metric]!
Saving new checkpoint.
Validation result at epoch   1, step    


>>> CONTROLLO  best dev WER = 38.86  | params: {'lr': 4.7377279007281495e-05}


## 5 — Study 2: DISTILLAZIONE (Optuna cerca `lambda`, `high_w`, `lr`)

In [20]:
def obj_distill(trial):
    lam    = trial.suggest_float('lambda', STUDY['lambda_min'], STUDY['lambda_max'], log=True)
    high_w = trial.suggest_float('high_w', STUDY['high_w_min'], STUDY['high_w_max'])
    lr     = trial.suggest_float('lr', STUDY['lr_min'], STUDY['lr_max'], log=True)
    cfg = build_cfg(enabled=True, lr=lr, lam=lam, low_w=1.0, high_w=high_w, tag=f"dist_{trial.number}")
    return train_return_wer(cfg)

study_distill = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=STUDY['seed']))
study_distill.optimize(obj_distill, n_trials=STUDY['n_trials_distill'])
best_distill = study_distill.best_value
print(f"\n>>> DISTILL   best dev WER = {best_distill:.2f}  | params: {study_distill.best_params}")

[I 2026-07-04 23:16:49,102] A new study created in memory with name: no-name-e87aeb45-77fa-49be-b855-951be4e82d6f
Loading model from ./distillation/student_40.ckpt
Reset optimizer.
Reset scheduler.
Reset tracking of the best checkpoint.


COPE:  False
[builders] SophiaG non disponibile, uso Adam come fallback


Distillation ON | student_dim=256 teacher_dim=512 | lambda=0.357 low_w=1.00 high_w=0.95 | N feats=7096
EPOCH 1
[Epoch: 001 Step: 00007200] Batch Recognition Loss:   1.350135 => Gls Tokens per Sec:     2436 || Lr: 0.000108
Hooray! New best validation result [eval_metric]!
Saving new checkpoint.
Validation result at epoch   1, step     7200: duration: 13.2125s
	Recognition Beam Size: 1	Translation Beam Size: -1	Translation Beam Alpha: -1
	Recognition Loss: 36.16757	Translation Loss: -1.00000	PPL: -1.00000
	Eval Metric: WER
	WER 39.44	(DEL: 18.36,	INS: 1.63,	SUB: 19.46)
	BLEU-4 -1.00	(BLEU-1: -1.00,	BLEU-2: -1.00,	BLEU-3: -1.00,	BLEU-4: -1.00)
	CHRF -1.00	ROUGE -1.00
Logging Recognition and Translation Outputs
Logging Sequence: 27September_2009_Sunday_tagesschau-2169
	Gloss Reference :	SUED SCHWACH WEHEN IX      MAESSIG KUESTE FRISCH OST STURM
	Gloss Hypothesis:	SUED SCHWACH ***** MAESSIG NORD    KUESTE FRISCH OST STURM
	Gloss Alignment :	             D     S       S                      

COPE:  False
[builders] SophiaG non disponibile, uso Adam come fallback


Distillation ON | student_dim=256 teacher_dim=512 | lambda=0.766 low_w=1.00 high_w=0.16 | N feats=7096
EPOCH 1
[Epoch: 001 Step: 00007200] Batch Recognition Loss:   1.204816 => Gls Tokens per Sec:     2449 || Lr: 0.000029
Hooray! New best validation result [eval_metric]!
Saving new checkpoint.
Validation result at epoch   1, step     7200: duration: 13.4451s
	Recognition Beam Size: 1	Translation Beam Size: -1	Translation Beam Alpha: -1
	Recognition Loss: 36.27549	Translation Loss: -1.00000	PPL: -1.00000
	Eval Metric: WER
	WER 39.79	(DEL: 18.87,	INS: 1.79,	SUB: 19.14)
	BLEU-4 -1.00	(BLEU-1: -1.00,	BLEU-2: -1.00,	BLEU-3: -1.00,	BLEU-4: -1.00)
	CHRF -1.00	ROUGE -1.00
Logging Recognition and Translation Outputs
Logging Sequence: 30May_2011_Monday_heute-3677
	Gloss Reference :	OST WARM REGION BIS DREISSIG GRAD REGION UEBERSPRINGEN REGION KUEHL NUR     FUENFZEHN SECHSZEHN GRAD
	Gloss Hypothesis:	OST WARM ****** *** ******** **** ****** DANN          NORD   KUEHL DEUTSCH LAND      SECHSZEHN *

...

COPE:  False
[builders] SophiaG non disponibile, uso Adam come fallback


Distillation ON | student_dim=256 teacher_dim=512 | lambda=1.300 low_w=1.00 high_w=0.45 | N feats=7096
EPOCH 1
[Epoch: 001 Step: 00007200] Batch Recognition Loss:   1.314727 => Gls Tokens per Sec:     2478 || Lr: 0.000020
Hooray! New best validation result [eval_metric]!
Saving new checkpoint.
Validation result at epoch   1, step     7200: duration: 13.4530s
	Recognition Beam Size: 1	Translation Beam Size: -1	Translation Beam Alpha: -1
	Recognition Loss: 36.29456	Translation Loss: -1.00000	PPL: -1.00000
	Eval Metric: WER
	WER 39.71	(DEL: 18.36,	INS: 1.87,	SUB: 19.48)
	BLEU-4 -1.00	(BLEU-1: -1.00,	BLEU-2: -1.00,	BLEU-3: -1.00,	BLEU-4: -1.00)
	CHRF -1.00	ROUGE -1.00
Logging Recognition and Translation Outputs
Logging Sequence: 07June_2010_Monday_heute-2322
	Gloss Reference :	DAZU WARM LUFT TATSAECHLICH WARM HEISS NAECHSTE-WOCHE KUEHL  
	Gloss Hypothesis:	DAZU WARM LUFT TATSAECHLICH **** ***** FREUNDLICH     SUEDOST
	Gloss Alignment :	                            D    D     S              

COPE:  False
[builders] SophiaG non disponibile, uso Adam come fallback


Distillation ON | student_dim=256 teacher_dim=512 | lambda=1.511 low_w=1.00 high_w=0.45 | N feats=7096
EPOCH 1
[Epoch: 001 Step: 00007200] Batch Recognition Loss:   1.535899 => Gls Tokens per Sec:     2514 || Lr: 0.000021
Hooray! New best validation result [eval_metric]!
Saving new checkpoint.
Validation result at epoch   1, step     7200: duration: 13.3964s
	Recognition Beam Size: 1	Translation Beam Size: -1	Translation Beam Alpha: -1
	Recognition Loss: 36.32643	Translation Loss: -1.00000	PPL: -1.00000
	Eval Metric: WER
	WER 39.95	(DEL: 18.41,	INS: 1.89,	SUB: 19.64)
	BLEU-4 -1.00	(BLEU-1: -1.00,	BLEU-2: -1.00,	BLEU-3: -1.00,	BLEU-4: -1.00)
	CHRF -1.00	ROUGE -1.00
Logging Recognition and Translation Outputs
Logging Sequence: 11December_2009_Friday_tagesschau-3509
	Gloss Reference :	ALPEN LANG SCHNEE
	Gloss Hypothesis:	***** **** ABEND 
	Gloss Alignment :	D     D    S     
[Epoch: 001 Step: 00007300] Batch Recognition Loss:   1.425135 => Gls Tokens per Sec:     2470 || Lr: 0.000021
Hoor


>>> DISTILL   best dev WER = 38.70  | params: {'lambda': 1.696753360719655, 'high_w': 0.21233911067827616, 'lr': 3.0398696602619608e-05}


In [21]:
import torch

def _safe_reduce_lr(self, epoch=None):
    # reimplementa _reduce_lr in modo sicuro: tollera min_lrs più corto dei param_groups
    for i, pg in enumerate(self.optimizer.param_groups):
        old_lr = float(pg["lr"])
        min_lr = self.min_lrs[i] if i < len(self.min_lrs) else 0.0
        new_lr = max(old_lr * self.factor, min_lr)
        if old_lr - new_lr > self.eps:
            pg["lr"] = new_lr

torch.optim.lr_scheduler.ReduceLROnPlateau._reduce_lr = _safe_reduce_lr
print("scheduler patch OK (self-contained, no recursion)")

scheduler patch OK (self-contained, no recursion)


## 6 — Verdetto + tabella riassuntiva

In [22]:
margin = STUDY['margin']
print('='*64)
print(f"CONTROLLO   (finetuning, no distill):  {best_ctrl:.2f}")
print(f"DISTILL     (FD-CMKD feature-level):   {best_distill:.2f}")
print(f"baseline student_40 (dev):             40.30")
print('-'*64)
delta = best_ctrl - best_distill
if best_distill < best_ctrl - margin:
    print(f"VERDETTO: la DISTILLAZIONE AIUTA  (delta {delta:+.2f} > margine {margin}).")
    print(f"  params migliori: {study_distill.best_params}")
    print("  -> tieni questa config, poi valuta sul TEST con beam search.")
else:
    print(f"VERDETTO: distillazione NEUTRA vs controllo (delta {delta:+.2f} <= margine {margin}).")
    print("  -> la feature-only non basta. PIANO B: shared classifier / logit distillation")
    print("     (richiede vocab condiviso). Vedi cella successiva.")
print('='*64)

# salva i risultati
import pandas as pd
rows = []
for name, st in [('ctrl', study_ctrl), ('distill', study_distill)]:
    for t in st.trials:
        rows.append({'study': name, 'trial': t.number, 'wer': t.value, **t.params})
df = pd.DataFrame(rows).sort_values('wer')
df.to_csv(os.path.join(STUDY['out_root'], 'optuna_results.csv'), index=False)
df.head(12)

CONTROLLO   (finetuning, no distill):  38.86
DISTILL     (FD-CMKD feature-level):   38.70
baseline student_40 (dev):             40.30
----------------------------------------------------------------
VERDETTO: distillazione NEUTRA vs controllo (delta +0.16 <= margine 0.3).
  -> la feature-only non basta. PIANO B: shared classifier / logit distillation
     (richiede vocab condiviso). Vedi cella successiva.


,study,trial,wer,lr,lambda,high_w
10,distill,4,38.697625,0.000030,1.696753,0.212339
19,distill,13,38.697625,0.000020,1.299524,0.448571
20,distill,14,38.751001,0.000021,1.510581,0.454751
11,distill,5,38.804377,0.000067,0.186600,0.304242
13,distill,7,38.804377,0.000046,0.160712,0.292145
14,distill,8,38.857753,0.000032,0.471705,0.785176
0,ctrl,0,38.857753,0.000047,NaN,NaN
4,ctrl,4,38.884441,0.000029,NaN,NaN
2,ctrl,2,38.911129,0.000108,NaN,NaN
5,ctrl,5,38.937817,0.000029,NaN,NaN
